# 🌿 Week 3: Git (first half) + NumPy Data Handling (second half)

This week has two parts:

| Days | Part | Why it matters |
|---|---|---|
| 1–2.5 | **Part A: Git** | Save versions of your work, undo mistakes, and collaborate. Every developer uses it daily. |
| 2.5–5 | **Part B: NumPy** | The foundation of data work in AI. Fast arrays instead of slow Python lists. |

**📖 Official docs (read a little every day):**
- Git: [Pro Git book (free)](https://git-scm.com/book/en/v2) – start with chapters 1 and 2 · [Git reference](https://git-scm.com/docs)
- GitHub: [Hello World guide](https://docs.github.com/en/get-started/start-your-journey/hello-world)
- NumPy: [NumPy: the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html) · [NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html)

Run the setup cell below first.

In [ ]:
# Run this cell first! It sets up a small helper that checks your exercise answers.
def check(test, hint=""):
    """Runs test() and prints a friendly pass / not-yet message."""
    try:
        ok = bool(test())
    except Exception as err:
        ok = False
        hint = f"{hint} (Python said: {type(err).__name__}: {err})"
    print("✅ Correct! Nice work." if ok else f"❌ Not yet. {hint}")

print("Setup done. You are ready to go!")
import os, shlex, shutil, stat, subprocess
import numpy as np

PLAYGROUND = "git_playground"   # a practice folder, safe to delete any time

def _force_remove(func, path, _exc):
    os.chmod(path, stat.S_IWRITE)   # Windows keeps .git files read-only
    func(path)

def git(command):
    """Run a git command inside the playground folder and print its output.
    Example: git("status")"""
    result = subprocess.run(["git"] + shlex.split(command), cwd=PLAYGROUND,
                            capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    print(f"$ git {command}")
    if output:
        print(output)
    return output

def git_quiet(command):
    """Same as git() but returns the output without printing it."""
    result = subprocess.run(["git"] + shlex.split(command), cwd=PLAYGROUND,
                            capture_output=True, text=True)
    return (result.stdout + result.stderr).strip()

def write_file(name, text):
    """Create (or overwrite) a file inside the playground folder."""
    with open(os.path.join(PLAYGROUND, name), "w", encoding="utf-8") as f:
        f.write(text)

def reset_playground():
    """Delete the playground and create a fresh empty git repository."""
    if os.path.exists(PLAYGROUND):
        shutil.rmtree(PLAYGROUND, onerror=_force_remove)
    os.makedirs(PLAYGROUND)
    git_quiet("init")
    git_quiet("checkout -b main")
    git_quiet("config user.name Learner")            # only for this practice repo (local config)
    git_quiet("config user.email learner@example.com")
    print("Fresh playground ready ->", os.path.abspath(PLAYGROUND))

---
# Part A: Git fundamentals

## What is Git and why use it?
Imagine writing an essay and saving `essay_final.docx`, `essay_final_v2.docx`, `essay_REAL_final.docx`... 😅
**Git** does that properly: it remembers every saved version (called a **commit**), who made it and why. You can go back in time, try ideas safely on a **branch**, and work with other people.

> **Git** is the tool on your computer. **GitHub** is a website that stores Git projects online. They are different things.

### The three places in Git
```
 Working folder  --git add-->  Staging area  --git commit-->  Repository (history)
 (your files)                 (ready to save)                 (saved versions)
```

### Install and first-time setup
Install from [git-scm.com/downloads](https://git-scm.com/downloads). Then run these **once** in a terminal (use your own name and email):
```bash
git --version
git config --global user.name  "Your Name"
git config --global user.email "you@example.com"
git config --global init.defaultBranch main
```
Run this cell to check Git is installed:

In [ ]:
print(subprocess.run(["git", "--version"], capture_output=True, text=True).stdout)

## A safe practice area
This notebook creates a folder called `git_playground` and runs Git commands inside it for you, using the helper `git("...")`. Nothing here can damage your real projects.

> Later, repeat these same commands **in your own terminal** inside a real folder. Typing them yourself is what builds the habit.

In [ ]:
reset_playground()   # run this again any time you want to start over

## Day 1: `init`, `status`, `add`, `commit`, `log`

| Command | What it does |
|---|---|
| `git init` | start tracking a folder (already done for you above) |
| `git status` | what changed? what is staged? |
| `git add <file>` | put a file in the staging area |
| `git commit -m "message"` | save a snapshot with a message |
| `git log --oneline` | list saved snapshots |

Write commit messages that finish the sentence "This commit will... ", e.g. `Add readme file`.

In [ ]:
write_file("README.md", "# My practice project\n")
git("status")

In [ ]:
git("add README.md")
git("status")

In [ ]:
git('commit -m "Add readme"')
git("log --oneline")

> Our helper `git(...)` runs the same command you would type in a terminal. Use single quotes outside and double quotes around the commit message.

### See what changed with `git diff`
Edit a file, then compare. Lines starting with `+` were added and `-` were removed.

In [ ]:
write_file("README.md", "# My practice project\nThis is my first Git project.\n")
git("diff")

### 🏋️ Exercise 1: Make your second commit

In the playground:
1. Create a file `hello.py` containing `print('hi')` using `write_file`.
2. Stage it with `git add`.
3. Commit it (e.g. `git('commit -m "Add hello"')`).
4. Also stage and commit the edited `README.md` from above.

After this the log should have **at least 3 commits**.

In [ ]:
# YOUR CODE HERE
# write_file(...)
# git("add ...")
# git("commit -m ...")

# ---- self-check (don't edit) ----
log = git_quiet('log --oneline')
check(lambda: len(log.splitlines()) >= 3 and 'hello.py' in git_quiet('ls-files'), 'You need 3+ commits and hello.py must be tracked (git ls-files lists tracked files)')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
write_file('hello.py', "print('hi')\n")
git('add hello.py')
git('commit -m "Add hello"')
git('add README.md')
git('commit -m "Update readme"')
```

</details>

### `.gitignore`: files Git should ignore
Some files should never be saved: secrets, big data files, temporary files. List them in a file called `.gitignore`:
```
__pycache__/
*.csv
.env
```

In [ ]:
write_file(".gitignore", "*.log\n__pycache__/\n")
write_file("debug.log", "temporary stuff\n")
git("status --short")      # debug.log is NOT listed. Git ignores it

## Day 2: Branches and merging

A **branch** is a separate line of work. The default is `main`. Create a branch for each new idea, then **merge** it back when it is good. If it goes wrong, just delete the branch.

```
main:     A---B---C-------F   (merge)
                   \     /
feature:            D---E
```

| Command | What it does |
|---|---|
| `git branch` | list branches |
| `git switch -c name` | create and move to a new branch |
| `git switch name` | move to an existing branch |
| `git merge name` | bring `name` into the current branch |
| `git branch -d name` | delete a merged branch |

In [ ]:
git("add .gitignore")
git('commit -m "Add gitignore"')
git("switch -c feature-docs")
write_file("docs.md", "# Docs\n")
git("add docs.md")
git('commit -m "Add docs"')
git("branch")

In [ ]:
git("switch main")
print("docs.md exists on main?", os.path.exists(os.path.join(PLAYGROUND, "docs.md")))   # False! It is only on the branch
git("merge feature-docs")
print("docs.md exists on main?", os.path.exists(os.path.join(PLAYGROUND, "docs.md")))   # True

### 🏋️ Exercise 2: Branch and merge

1. Create a branch called `feature-greeting`.
2. On it, create `greeting.txt` with some text, stage and commit it.
3. Switch back to `main` and merge `feature-greeting`.

You should end on `main` with `greeting.txt` present.

In [ ]:
# YOUR CODE HERE

# ---- self-check (don't edit) ----
check(lambda: git_quiet('branch --show-current') == 'main' and os.path.exists(os.path.join(PLAYGROUND, 'greeting.txt')) and 'feature-greeting' in git_quiet('branch --merged'), 'Be on main, greeting.txt must exist, and feature-greeting must be merged')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
git('switch -c feature-greeting')
write_file('greeting.txt', 'Hello from a branch\n')
git('add greeting.txt')
git('commit -m "Add greeting"')
git('switch main')
git('merge feature-greeting')
```

</details>

## Day 2 (continued): Remotes, GitHub, conflicts, undoing

These need a real terminal and a GitHub account, so do them by hand. Treat this as your **hands-on checklist**:

### 🧪 Terminal task 1: Put your work on GitHub
1. Create a free account at [github.com](https://github.com).
2. Create a **new empty repository** on GitHub (e.g. `my-ai-learning`). Do not add a README there.
3. In your terminal, make a folder, `git init`, add a file, commit.
4. Connect and upload:
```bash
git remote add origin https://github.com/<your-username>/my-ai-learning.git
git branch -M main
git push -u origin main
```
5. Edit a file on the GitHub website, then run `git pull` locally and see the change arrive.

### 🧪 Terminal task 2: Create and fix a merge conflict
A **conflict** happens when two branches change the same line. Git asks *you* to decide.
1. On `main` put `color = "blue"` in `settings.txt` and commit.
2. `git switch -c experiment`, change the line to `color = "red"`, commit.
3. `git switch main`, change the same line to `color = "green"`, commit.
4. `git merge experiment` -> Git reports a conflict. Open the file:
```
<<<<<<< HEAD
color = "green"
=======
color = "red"
>>>>>>> experiment
```
5. Edit the file to the final version you want, remove the `<<<`, `===`, `>>>` lines, then `git add settings.txt` and `git commit`.

### 🧪 Terminal task 3: Contribute like a pro (fork and pull request)
1. **Fork** this learning repository on GitHub (button top-right).
2. `git clone` your fork, create a branch `my-name-week3`, add a file `students/<your-name>.md`.
3. Commit, push the branch, and open a **Pull Request** on GitHub.

### Handy "oops" commands
| Situation | Command |
|---|---|
| I edited a file and want to throw the change away | `git restore <file>` |
| I staged a file by mistake | `git restore --staged <file>` |
| What did I do recently? | `git log --oneline --graph` |
| Download a project | `git clone <url>` |
| Get latest changes | `git pull` |

📖 Docs: [Git Basics (Pro Git ch. 2)](https://git-scm.com/book/en/v2/Git-Basics-Getting-a-Git-Repository), [Branching (ch. 3)](https://git-scm.com/book/en/v2/Git-Branching-Branches-in-a-Nutshell), [GitHub Docs](https://docs.github.com/en/get-started)

### 🏋️ Exercise 3: Undo a mistake

The first line below overwrites `README.md` with the wrong text (a pretend accident). Use `git status` to see it, then use `git restore README.md` to bring back the committed version. The file should no longer contain the word `oops`.

In [ ]:
write_file('README.md', 'oops, wrong!\n')   # the accident
# YOUR CODE HERE

# ---- self-check (don't edit) ----
readme = open(os.path.join(PLAYGROUND, 'README.md'), encoding='utf-8').read()
check(lambda: 'oops' not in readme and 'practice project' in readme, 'git restore README.md should bring the saved text back')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
write_file('README.md', 'oops, wrong!\n')
git('restore README.md')
```

</details>

### 🎯 Git habit for the rest of this course
After each study day, commit your notebook work in your own copy (fork) of this repo:
```bash
git add .
git commit -m "Finish week 3 day 3 exercises"
git push
```
Small, regular commits are the best way to practise.

---
# Part B: NumPy data handling

## Why NumPy?
Python lists are flexible but slow for maths. **NumPy** gives us the `ndarray`: a fast grid of numbers of the same type. AI and data science libraries (pandas, scikit-learn, PyTorch) are built on top of it, and in Week 4 you will build machine learning models with only NumPy.

📖 Docs: [NumPy: the absolute basics](https://numpy.org/doc/stable/user/absolute_beginners.html)

By convention we import it as `np` (we already did in the setup cell, but it is good to see it):

In [ ]:
import numpy as np
print(np.__version__)

## Day 3: Arrays: creating and understanding them

In [ ]:
a = np.array([10, 20, 30, 40])        # from a list
m = np.array([[1, 2, 3],
              [4, 5, 6]])              # 2D array: 2 rows, 3 columns

print(a, a.dtype)      # dtype = data type of the elements
print(m.shape)         # (2, 3) -> (rows, columns)
print(m.ndim, m.size)  # number of dimensions, total elements

### Handy ways to create arrays

In [ ]:
print(np.zeros((2, 3)))          # all zeros
print(np.ones(4))                # all ones
print(np.arange(0, 10, 2))       # like range: 0,2,4,6,8
print(np.linspace(0, 1, 5))      # 5 evenly spaced numbers from 0 to 1
print(np.full((2, 2), 7))        # filled with 7

rng = np.random.default_rng(42)   # random generator with a seed (same numbers every time)
print(rng.random(3))              # 3 random floats in [0, 1)
print(rng.integers(1, 7, size=5)) # 5 dice rolls

**Try it:** change the seed 42 to 7 and run again. What changes?

### 🏋️ Exercise 4: Create arrays

1. `evens`: all even numbers from 2 to 20 (use `np.arange`).
2. `grid`: a 3×4 array containing numbers 0 to 11 (use `np.arange(12)` then `.reshape(3, 4)`).
3. `zeros_int`: a 2×2 array of zeros whose dtype is integer (hint: `np.zeros((2, 2), dtype=int)`).

In [ ]:
evens = None
grid = None
zeros_int = None

# ---- self-check (don't edit) ----
check(lambda: list(evens) == [2,4,6,8,10,12,14,16,18,20] and grid.shape == (3, 4) and grid[2, 3] == 11 and zeros_int.dtype.kind == 'i', 'evens 2..20, grid 3x4 with values 0..11, integer zeros')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
evens = np.arange(2, 21, 2)
grid = np.arange(12).reshape(3, 4)
zeros_int = np.zeros((2, 2), dtype=int)
```

</details>

### Indexing and slicing
Same ideas as lists, but with a comma for each dimension: `array[row, column]`. A colon `:` means "all".

In [ ]:
grid = np.arange(12).reshape(3, 4)
print(grid)
print(grid[1, 2])      # row 1, column 2 -> 6
print(grid[0])         # first row
print(grid[:, 1])      # all rows, column 1
print(grid[1:, :2])    # rows 1.., columns 0..1

### 🏋️ Exercise 5: Indexing

Using `data` below, store:
- `second_row` -> the whole second row
- `last_col` -> the last column
- `corner` -> the 2×2 block from the bottom-right corner

In [ ]:
data = np.arange(1, 17).reshape(4, 4)
second_row = None
last_col = None
corner = None

# ---- self-check (don't edit) ----
check(lambda: list(second_row) == [5,6,7,8] and list(last_col) == [4,8,12,16] and corner.tolist() == [[11,12],[15,16]], 'second_row [5,6,7,8]; last_col [4,8,12,16]; corner [[11,12],[15,16]]')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
data = np.arange(1, 17).reshape(4, 4)
second_row = data[1]
last_col = data[:, -1]
corner = data[-2:, -2:]
```

</details>

## Day 4: Maths without loops: vectorization, broadcasting, masks

### Vectorized operations
Operations apply to **every element at once**, with no `for` loop. Much shorter and much faster.

In [ ]:
prices = np.array([100, 250, 80, 400])
print(prices * 1.18)           # add 18% tax to all
print(prices + prices)
print(prices > 200)            # array of True / False
print(np.sqrt(prices), prices ** 2)

### Broadcasting
NumPy can combine arrays of **different shapes** by stretching the smaller one, as long as the shapes are compatible.

In [ ]:
table = np.array([[1, 2, 3],
                  [4, 5, 6]])
print(table + 10)                      # a number is stretched to every cell
print(table + np.array([100, 200, 300]))   # the row is added to every row

📖 Docs: [Broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html)

### Boolean masks: filter with conditions

In [ ]:
temps = np.array([28, 35, 31, 22, 39, 30])
mask = temps > 30
print(mask)
print(temps[mask])           # only the hot days
print((temps > 30).sum())    # how many hot days (True counts as 1)
temps[temps < 25] = 25       # change values by condition
print(temps)

### 🏋️ Exercise 6: Vectorize it

1. `fahrenheit`: convert `celsius` to Fahrenheit with `c * 9 / 5 + 32` for all values at once (no loops).
2. `hot`: only the Celsius values above 30.
3. `hot_count`: how many values are above 30.

In [ ]:
celsius = np.array([20.0, 35.5, 31.0, 18.2, 40.0])
fahrenheit = None
hot = None
hot_count = None

# ---- self-check (don't edit) ----
check(lambda: np.allclose(fahrenheit, [68, 95.9, 87.8, 64.76, 104]) and list(hot) == [35.5, 31.0, 40.0] and hot_count == 3, 'F = C*9/5+32 for all; hot = [35.5, 31.0, 40.0]; count 3')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
celsius = np.array([20.0, 35.5, 31.0, 18.2, 40.0])
fahrenheit = celsius * 9 / 5 + 32
hot = celsius[celsius > 30]
hot_count = (celsius > 30).sum()
```

</details>

### Aggregations and the `axis` idea
`sum`, `mean`, `min`, `max`, `std`... can work on everything, or per column / per row:
- `axis=0` -> go **down** the rows (one result per **column**)
- `axis=1` -> go **across** the columns (one result per **row**)

In [ ]:
scores = np.array([[80, 90, 70],
                   [60, 75, 85]])
print(scores.mean())           # everything
print(scores.mean(axis=0))     # per column: [70. 82.5 77.5]
print(scores.mean(axis=1))     # per row: [80. 73.33]
print(scores.max(axis=1), scores.argmax(axis=1))   # biggest value and its position

### 🏋️ Exercise 7: Aggregations

For the 3 students (rows) × 4 subjects (columns) in `marks`, compute:
- `per_student` -> each student's average (one value per row)
- `per_subject` -> each subject's average (one value per column)
- `best_student` -> index of the student with the highest average (`argmax`)

In [ ]:
marks = np.array([[70, 80, 90, 60],
                  [95, 85, 90, 100],
                  [50, 60, 55, 65]])
per_student = None
per_subject = None
best_student = None

# ---- self-check (don't edit) ----
check(lambda: np.allclose(per_student, [75, 92.5, 57.5]) and np.allclose(per_subject, [71.6667, 75, 78.3333, 75], atol=1e-3) and best_student == 1, 'per_student [75, 92.5, 57.5]; best_student 1')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
marks = np.array([[70, 80, 90, 60],
                  [95, 85, 90, 100],
                  [50, 60, 55, 65]])
per_student = marks.mean(axis=1)
per_subject = marks.mean(axis=0)
best_student = per_student.argmax()
```

</details>

## Day 5: Reshaping, combining, missing data, loading files

### Reshape, transpose, stack

In [ ]:
x = np.arange(6)
print(x.reshape(2, 3))
print(x.reshape(3, -1))      # -1 means "figure it out"
y = x.reshape(2, 3)
print(y.T)                   # transpose: swap rows and columns
print(np.vstack([y, y]))     # stack vertically
print(np.hstack([y, y]))     # stack horizontally
print(np.sort([3, 1, 2]), np.unique([1, 1, 2, 3, 3]))

### Missing values: `np.nan`
Real data has holes. `np.nan` means "not a number". Careful: any maths with `nan` gives `nan`. Use `np.isnan` to find them and `np.nanmean` to ignore them.

In [ ]:
ages = np.array([21.0, np.nan, 25.0, 30.0, np.nan])
print(ages.mean())              # nan
print(np.isnan(ages))
print(np.nanmean(ages))         # ignores nan -> 25.33
ages[np.isnan(ages)] = np.nanmean(ages)   # fill the holes with the average
print(ages)

### 🏋️ Exercise 8: Clean the data

`readings` has missing values (`np.nan`). Create `cleaned` where every `nan` is replaced by the **mean of the non-missing values**. Also store `missing_count` = number of missing values.

In [ ]:
readings = np.array([10.0, np.nan, 14.0, 12.0, np.nan, 16.0])
cleaned = None
missing_count = None

# ---- self-check (don't edit) ----
check(lambda: missing_count == 2 and np.allclose(cleaned, [10, 13, 14, 12, 13, 16]), 'The mean of 10,14,12,16 is 13, so nan -> 13; missing_count = 2')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
readings = np.array([10.0, np.nan, 14.0, 12.0, np.nan, 16.0])
missing_count = np.isnan(readings).sum()
cleaned = readings.copy()
cleaned[np.isnan(cleaned)] = np.nanmean(readings)
```

</details>

### Loading a CSV file
A CSV is a text file with values separated by commas. `np.genfromtxt` reads it (and handles gaps as `nan`).
Let's create a tiny file and read it back:

In [ ]:
csv_text = """height,weight,age
170,65,21
160,,25
182,80,30
"""
with open("people_demo.csv", "w") as f:
    f.write(csv_text)

people = np.genfromtxt("people_demo.csv", delimiter=",", skip_header=1)
print(people)
print(people.shape)

📖 Docs: [`numpy.genfromtxt`](https://numpy.org/doc/stable/reference/generated/numpy.genfromtxt.html) and [Reading and writing files](https://numpy.org/doc/stable/user/how-to-io.html)

### 🏋️ Exercise 9: Load and normalise

Using `people` (loaded above):
1. `col_means` -> mean of every column ignoring `nan` (`np.nanmean(..., axis=0)`).
2. `filled` -> a copy of `people` where `nan` is replaced by its column mean (hint: `np.where(np.isnan(people), col_means, people)` uses broadcasting!).
3. `scaled` -> `filled` with each column scaled to have mean 0 and std 1: `(filled - mean) / std` per column.

In [ ]:
col_means = None
filled = None
scaled = None

# ---- self-check (don't edit) ----
check(lambda: not np.isnan(filled).any() and filled[1, 1] == 72.5 and np.allclose(scaled.mean(axis=0), 0) and np.allclose(scaled.std(axis=0), 1), 'No nan left; the missing weight becomes 72.5; each scaled column has mean 0 and std 1')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
col_means = np.nanmean(people, axis=0)
filled = np.where(np.isnan(people), col_means, people)
scaled = (filled - filled.mean(axis=0)) / filled.std(axis=0)
```

</details>

## 🚀 Mini project: Class marks analyser
You have the marks of 6 students in 4 subjects. Write one function `analyse(marks)` that returns a dictionary with:

| Key | Meaning |
|---|---|
| `"averages"` | each student's average (array of 6) |
| `"ranking"` | student indexes from best to worst average (hint: `np.argsort`, then reverse with `[::-1]`) |
| `"passed_all"` | indexes of students who scored **40 or more in every subject** (hint: `(marks >= 40).all(axis=1)` and `np.where`) |
| `"subject_toppers"` | for each subject, the index of the student with the highest mark |
| `"normalized"` | marks scaled per subject to the range 0–1: `(x - min) / (max - min)` using `axis=0` |

Use **no `for` loops**. Make NumPy do the work.

### 🏋️ Exercise 10: Marks analyser

Complete `analyse`.

In [ ]:
def analyse(marks):
    # YOUR CODE HERE
    pass

# ---- self-check (don't edit) ----
def _test():
    marks = np.array([[70, 80, 90, 60],
                      [95, 85, 90, 100],
                      [50, 60, 55, 65],
                      [30, 90, 85, 70],
                      [88, 70, 60, 75],
                      [45, 55, 65, 35]])
    r = analyse(marks)
    return (np.allclose(r["averages"], [75, 92.5, 57.5, 68.75, 73.25, 50])
            and list(r["ranking"]) == [1, 0, 4, 3, 2, 5]
            and list(r["passed_all"]) == [0, 1, 2, 4]
            and list(r["subject_toppers"]) == [1, 3, 0, 1]
            and r["normalized"].min() == 0 and r["normalized"].max() == 1)
check(_test, 'Check averages, ranking (best first), passed_all, subject_toppers and normalized (all values between 0 and 1)')

<details>
<summary>💡 Stuck? Click to see one possible solution (try first!)</summary>

```python
def analyse(marks):
    averages = marks.mean(axis=1)
    ranking = np.argsort(averages)[::-1]
    passed_all = np.where((marks >= 40).all(axis=1))[0]
    subject_toppers = marks.argmax(axis=0)
    mn, mx = marks.min(axis=0), marks.max(axis=0)
    normalized = (marks - mn) / (mx - mn)
    return {"averages": averages, "ranking": ranking, "passed_all": passed_all,
            "subject_toppers": subject_toppers, "normalized": normalized}
```

</details>

## ✅ Week 3 checklist
**Git**
- [ ] I can explain working folder, staging area and repository
- [ ] I can `init`, `status`, `add`, `commit`, `log`, `diff`
- [ ] I can create a branch, switch, and merge
- [ ] I pushed a project to GitHub and made a pull request
- [ ] I fixed a merge conflict at least once
- [ ] I use `.gitignore`

**NumPy**
- [ ] I can create arrays (`array`, `zeros`, `arange`, `linspace`, random)
- [ ] I understand `shape`, `dtype`, indexing and slicing
- [ ] I can do vectorized maths and know the idea of broadcasting
- [ ] I can filter with boolean masks
- [ ] I use `axis` with `sum` / `mean` / `max`
- [ ] I can handle `nan` and load a CSV
- [ ] I finished the Marks analyser

## 📚 Read next
- [Pro Git book](https://git-scm.com/book/en/v2) chapters 1–3
- [NumPy quickstart](https://numpy.org/doc/stable/user/quickstart.html)
- [NumPy: the absolute basics for beginners](https://numpy.org/doc/stable/user/absolute_beginners.html)

➡️ Next: `week4_numpy_supervised_ml.ipynb`: you will use everything from this week on a real Kaggle dataset!